In [4]:
!pip freeze | grep scikit-learn

'grep' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
!python -V

Python 3.10.20


In [6]:
import os
import pickle
import pandas as pd

In [7]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [8]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df


In [9]:
df = read_data('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet')

In [10]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [11]:
print(f'Q1 -- std dev of predicted duration: {y_pred.std():.2f}')

Q1 -- std dev of predicted duration: 6.25


In [12]:
year = 2023
month = 3
output_file = f'output_{year:04d}-{month:02d}.parquet'

df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

df_result = pd.DataFrame({
    'ride_id': df['ride_id'],
    'predicted_duration': y_pred,
})

df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)

size_bytes = os.path.getsize(output_file)
print(f'Q2 -- output file size: {size_bytes} bytes ({size_bytes / 1024 / 1024:.2f} MB)')

Q2 -- output file size: 68558004 bytes (65.38 MB)


In [14]:
print('Q3 -- jupyter nbconvert --to script starter.ipynb')

Q3 -- jupyter nbconvert --to script starter.ipynb
